In [2]:
!pip install transformers==4.56.1 peft==0.17.0 accelerate==1.10.0 trl==0.23.1 datasets==4.0.0 huggingface-hub==0.34.4 safetensors==0.6.2 pandas==2.2.2 matplotlib==3.10.0 numpy==2.0.2

  Using cached transformers-4.56.1-py3-none-any.whl.metadata (42 kB)
  Using cached peft-0.17.0-py3-none-any.whl.metadata (14 kB)
  Using cached accelerate-1.10.0-py3-none-any.whl.metadata (19 kB)
  Using cached trl-0.23.1-py3-none-any.whl.metadata (11 kB)
  Using cached datasets-4.0.0-py3-none-any.whl.metadata (19 kB)
  Using cached huggingface_hub-0.34.4-py3-none-any.whl.metadata (14 kB)
  Using cached numpy-2.0.2-cp312-cp312-macosx_14_0_arm64.whl.metadata (60 kB)
  Using cached regex-2026.4.4-cp312-cp312-macosx_11_0_arm64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-macosx_11_0_arm64.whl.metadata (7.3 kB)
  Using cached torch-2.11.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (29 kB)
  Using cached pyarrow-24.0.0-cp312-cp312-macosx_12_0_arm64.whl.metadata (3.0 kB)
  Using cached dill-0.3.8-py3-none-any.whl.metadata (10 kB)
  Using cached xxhash-3.7.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (13 kB)
  Using cached multiprocess-0.70.16-py312-none-any.whl.metadata (

In [1]:
import os
import torch
from datasets import load_dataset
from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import json

/Users/sashakrigel/Documents/Spring_2026/6.C395 final/acc-alt-text/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
import datasets

### Helpers 

In [14]:
def build_prompt(scenegraph):

    PROMPT = f"""You are an accessibility helper.

    Please generate WCAG-aligned alt-text for the below data visualization:
    {scenegraph}
"""
    return PROMPT

In [15]:
def format_example(example,  target_feature="caption_L1"):
    prompt = build_prompt(example["scenegraph"])
    target_descr =example[target_feature]
    converted_sample = [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": target_descr},
        ]
    return {"messages":converted_sample}

### Load Data

In [16]:
import json

DATA_PATH = "../data/vistext_train_test/data_train.json"

with open(DATA_PATH) as f:
    train_data = json.load(f)

print(f"Loaded {len(train_data)} examples")
print("Keys:", list(train_data[0].keys()))

dataset = datasets.Dataset.from_list(train_data).map(lambda ex: format_example(ex, "caption_L1"))


Loaded 9969 examples
Keys: ['caption_id', 'img_id', 'split', 'scenegraph', 'datatable', 'caption_L1', 'caption_L2L3', 'L1_properties']


Map: 100%|██████████| 9969/9969 [00:00<00:00, 26289.97 examples/s]


### Load Models

In [ ]:
CANDIDATE_MODELS = [
    "Qwen/Qwen2.5-1.5B-Instruct",
    "microsoft/phi-2",                 
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
]

NUM_EXAMPLES = 50

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

In [ ]:
MODEL= CANDIDATE_MODELS[0]

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding


tokenizer = AutoTokenizer.from_pretrained(MODEL)


def tokenize_function(example):
    #TODO
    pass


tokenized_datasets = train_data.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)